# 04 — Tuning sensitivity (optional)

Tunes RandomForest, XGBoost and LSTM on the inner split (2016–2019 → 2020) and compares the
tuned models with the fixed values on 2021. Results go to `results_tuning/`. LSTM needs a GPU.

In [ ]:
from temporal_core import *

RESULTS = load_results(CFG)

In [ ]:
from dataclasses import replace
import itertools

CFG_TUNE = replace(
    CFG,
    results_folder_name="results_tuning",
    tune_algorithms=("RandomForest", "XGBoost"),
    tuning_subsample=None,
    param_grids={
        "RandomForest": [{"min_samples_leaf": l, "max_features": f}
                         for l in (1, 2, 5, 10) for f in (1.0, 0.5, "sqrt")],
        "XGBoost": [{"max_depth": d, "learning_rate": lr, "n_estimators": n}
                    for d in (4, 6, 8) for lr in (0.03, 0.05, 0.1) for n in (400, 800)],
    },
)
CASES = build_cases(CFG_TUNE)
TUNED = {alg: tune_on_training_years(alg, CASES, CFG_TUNE) for alg in ("RandomForest", "XGBoost")}

In [ ]:
# Second RandomForest grid: larger leaves
CFG_TUNE = replace(CFG_TUNE, param_grids={
    **CFG_TUNE.param_grids,
    "RandomForest": [{"min_samples_leaf": l, "max_features": 1.0} for l in (10, 20, 50, 100)]})
TUNED["RandomForest"] = tune_on_training_years("RandomForest", CASES, CFG_TUNE)
print(TUNED)

In [ ]:
def compare_with_main(alg, name, cfg_new):
    a = load_predictions(alg, CFG, name)
    b = load_predictions(alg, cfg_new, name)
    m = a.merge(b, on=CFG.time_col, suffixes=("_main", "_tuned"))
    m = m[m["Eval_Main_main"].astype(bool)]
    return daily_bootstrap((m["y_pred_tuned"] - m["y_true_main"]).to_numpy(),
                           (m["y_pred_main"] - m["y_true_main"]).to_numpy(),
                           m[CFG.time_col].dt.date.to_numpy(), CFG.bootstrap_n, CFG.random_seed)

rows = []
for alg, (best, _) in TUNED.items():
    for name, (tr, te) in CASES.items():
        r, _ = run_single_case(alg, name, tr, te, CFG_TUNE, best, quiet=True, with_importance=False)
        bs = compare_with_main(alg, name, CFG_TUNE)
        old = RESULTS[(RESULTS.Model == alg) & (RESULTS.Case == name)].iloc[0]
        rows.append({"Model": alg, "Case": name, "Params": str(best),
                     "RMSE_fixed": old.RMSE, "RMSE_tuned": r["RMSE"],
                     "dRMSE": bs["dRMSE"], "CI_lo": bs["dRMSE_lo"], "CI_hi": bs["dRMSE_hi"],
                     "p": bs["p_RMSE"]})
TUNING_RF_XGB = pd.DataFrame(rows)
print(TUNING_RF_XGB.round(4).to_string(index=False))

In [ ]:
# LSTM: inner grid (2 seeds per combination)
tr_long = CASES[CFG.case_long][0]
inner_tr = tr_long[tr_long[CFG.year_col].isin(CFG.inner_train_years)]
inner_va = tr_long[tr_long[CFG.year_col] == CFG.inner_validation_year]

rows = []
for units, epochs in itertools.product((32, 64, 128), (30, 60, 120)):
    c = replace(CFG_TUNE, lstm_units=units, lstm_epochs=epochs)
    for s in (42, 43):
        r2 = run_single_case("LSTM", "inner", inner_tr, inner_va, c, {}, quiet=True,
                             return_score_only=True, seed=s, with_importance=False)
        rows.append({"units": units, "epochs": epochs, "seed": s, "R2": r2})
LSTM_INNER = pd.DataFrame(rows)
print(LSTM_INNER.groupby(["units", "epochs"])["R2"].agg(["mean", "std"])
      .sort_values("mean", ascending=False).round(4))

In [ ]:
# LSTM: best inner combination on 2021 (five seeds, median-seed predictions)
c_best = replace(CFG_TUNE, lstm_units=128, lstm_epochs=120)
rows = []
for name, (tr, te) in CASES.items():
    runs = [run_single_case("LSTM", name, tr, te, c_best, {}, quiet=True, seed=s,
                            tag=f"_s{s}", with_importance=False)[0] for s in CFG.lstm_seeds]
    r2 = np.array([x["R2"] for x in runs])
    med = runs[int(np.argsort(r2)[len(r2) // 2])]["Seed"]
    pd.read_csv(os.path.join(c_best.predictions_dir, f"LSTM_{name}_s{med}.csv")).to_csv(
        os.path.join(c_best.predictions_dir, f"LSTM_{name}.csv"), index=False)
    old = RESULTS[(RESULTS.Model == "LSTM") & (RESULTS.Case == name)]
    bs = compare_with_main("LSTM", name, c_best)
    rows.append({"Case": name,
                 "R2_fixed": f"{old.R2.mean():.4f} ± {old.R2.std(ddof=1):.4f}",
                 "R2_tuned": f"{r2.mean():.4f} ± {r2.std(ddof=1):.4f}",
                 "dRMSE_median_seed": round(bs["dRMSE"], 4),
                 "CI": f"[{bs['dRMSE_lo']:.4f}, {bs['dRMSE_hi']:.4f}]", "p": round(bs["p_RMSE"], 4)})
TUNING_LSTM = pd.DataFrame(rows)
print(TUNING_LSTM.to_string(index=False))